# 🧠 Brainclip Complete Runtime — Colab

**One notebook to rule them all.** This notebook sets up the complete Brainclip backend on Google Colab:

| Capability | Stack |
|---|---|
| 🎵 Voice Synthesis (TTS) | Fish S2-Pro (5B params, zero-shot cloning) |
| 🎥 Video Rendering | Remotion + Headless Chromium + Node.js 18 |
| 🌐 Public API Tunnel | Cloudflare Tunnel (free, no account needed) |
| 🔍 Transcription | Faster-Whisper large-v2 (word-level timestamps) |

---

### Why an isolated virtual environment?

Colab’s base Python packages change without warning between updates. By pinning every
dependency inside a local `/content/venv`, the notebook survives Colab image releases
without silent breakage.

### GPU requirement

**T4 (16 GB VRAM) or better.** S2-Pro BF16 weights need ~10 GB. The remaining headroom
covers KV cache, audio buffers, and Whisper.

### How to use

1. **Runtime → Change runtime type → GPU (T4)**
2. **Run All** (Runtime → Run all)
3. Copy the printed **Cloudflare Tunnel URL** into Brainclip Settings → Colab Tunnel URL
4. Keep this notebook running — closing it kills the server

---
## ➀ Preflight: Python version check

Verify we are on Python 3.10+ (required by torch 2.6 and vLLM 0.8.4).

In [ ]:
import sys, os
from pathlib import Path

APP_ROOT = Path('/content')
VENV_DIR = APP_ROOT / 'venv'

print(f'Python version : {sys.version}')
print(f'Colab release  : {os.environ.get("COLAB_RELEASE_TAG", "unknown")}')

if sys.version_info < (3, 10):
    raise RuntimeError(
        f'Brainclip requires Python ≥ 3.10, but this Colab has {sys.version}. '
        'Try Runtime → Disconnect and delete runtime, then reconnect.'
    )
print('✅ Python version OK')

---
## ➁ System packages: Node.js 18, Chromium, FFmpeg

These are system-level (apt) dependencies needed by Remotion for headless video rendering.

In [ ]:
%%bash
set -e

echo '=== Installing Node.js 18 ==='
if command -v node &>/dev/null && [[ "$(node -v)" == v18* ]]; then
    echo "Node.js $(node -v) already installed — skipping."
else
    curl -fsSL https://deb.nodesource.com/setup_18.x | sudo -E bash - 2>&1 | tail -5
    sudo apt-get install -y nodejs 2>&1 | tail -3
fi

echo '=== Installing Chromium & FFmpeg ==='
sudo apt-get install -y chromium-browser ffmpeg portaudio19-dev python3-pyaudio 2>&1 | tail -5

echo ''
echo "✅ node $(node -v)  |  npm $(npm -v)  |  ffmpeg $(ffmpeg -version 2>&1 | head -1 | awk '{print $3}')"

---
## ➂ Clone Brainclip repo + create isolated virtual environment

We:
1. Clone the Brainclip repo (or skip if already present)
2. Create a fresh venv with `--without-pip` (avoids `ensurepip` errors on Colab)
3. Bootstrap pip manually
4. Pin build tools to known-good versions

**All subsequent `pip install` commands target the venv, never the system Python.**

In [ ]:
%%bash
set -e
cd /content

# ── Clone repo if not already present ──
if [ ! -d "notebooks" ]; then
    echo '=== Cloning Brainclip repository ==='
    git clone --depth 1 https://github.com/harshal0704/Brainclip.git temp_repo 2>&1 | tail -3
    # Move contents to /content root
    shopt -s dotglob
    mv temp_repo/* . 2>/dev/null || true
    rm -rf temp_repo
    echo '✅ Repository cloned'
else
    echo '✅ Repository already present'
fi

# ── Create fresh venv ──
echo '=== Creating isolated virtual environment ==='
rm -rf /content/venv
python3 -m venv /content/venv --without-pip

# Activate and bootstrap pip
source /content/venv/bin/activate
curl -sS https://bootstrap.pypa.io/get-pip.py | python 2>&1 | tail -3

# Pin build tools to known-good versions
python -m pip install --upgrade 'pip<25' 'setuptools<75' 'wheel<0.46' 2>&1 | tail -5

echo ''
echo "✅ venv created at /content/venv"
echo "   pip   : $(python -m pip --version | awk '{print $2}')"
echo "   setup : $(python -c 'import setuptools; print(setuptools.__version__)')"

---
## ➃ Install Python dependencies (pinned versions)

We install in a specific order to avoid dependency conflicts:

1. **PyTorch 2.6.0** — must match vLLM’s requirement exactly
2. **vLLM 0.8.4** — inference engine for S2-Pro, pulls correct transformers/xformers
3. **Application requirements** — FastAPI, Whisper, audio libs (with conflicting pins removed)
4. **Remotion npm install** — for video rendering

**Approximate install time:** 5–10 minutes (most packages are cached after first run).

### Pinned dependency manifest

| Package | Version | Reason |
|---|---|---|
| torch | 2.6.0 | vLLM 0.8.4 hard requirement |
| torchaudio | 2.6.0 | Must match torch version |
| vllm | 0.8.4 | S2-Pro inference backend |
| fastapi | 0.115.8 | Server framework |
| uvicorn | 0.34.0 | ASGI server |
| faster-whisper | 1.1.1 | Transcription |
| numpy | 2.1.3 | Stable numeric backend |
| scipy | 1.15.2 | Audio processing |
| pydantic | 2.10.6 | Data models |
| accelerate | 1.2.1 | HF model loading |
| soundfile | 0.13.1 | WAV I/O |
| loguru | 0.7.3 | Logging |
| transformers | 4.49.0 | HF transformers (5.4.0 breaks vLLM) |

In [ ]:
%%bash
set -e
source /content/venv/bin/activate

echo '=== [1/4] Installing PyTorch 2.6.0 + torchaudio ==='
python -m pip install --upgrade \
    'torch==2.6.0' \
    'torchaudio==2.6.0' \
    'huggingface-hub[hf_xet]>=0.30.0' \
    2>&1 | tail -5

echo ''
echo '=== [2/4] Installing vLLM 0.8.4 ==='
python -m pip install 'vllm==0.8.4' 2>&1 | tail -5

echo ''
echo '=== [3/4] Installing application requirements ==='

# Write a clean requirements file without packages that conflict with vLLM's pins
cat > /tmp/brainclip_requirements.txt << 'REQS'
fastapi==0.115.8
uvicorn[standard]==0.34.0
python-multipart==0.0.20
requests==2.32.3
soundfile==0.13.1
numpy==2.1.3
scipy==1.15.2
faster-whisper==1.1.1
sentencepiece==0.2.0
einops==0.8.1
loguru==0.7.3
accelerate==1.2.1
transformers==4.49.0
pydantic==2.10.6
nest-asyncio==1.6.0
fish-speech
REQS

python -m pip install -r /tmp/brainclip_requirements.txt 2>&1 | tail -10

echo ''
echo '=== [4/4] Installing Remotion npm dependencies ==='
cd /content/remotion && npm install 2>&1 | tail -5

echo ''
echo '=== Installed package summary ==='
python -c "
import importlib
pkgs = {
    'torch': 'torch', 'torchaudio': 'torchaudio',
    'vllm': 'vllm', 'fastapi': 'fastapi',
    'uvicorn': 'uvicorn', 'numpy': 'numpy',
    'scipy': 'scipy', 'pydantic': 'pydantic',
    'faster_whisper': 'faster_whisper',
    'transformers': 'transformers',
    'accelerate': 'accelerate',
    'soundfile': 'soundfile',
    'loguru': 'loguru',
}
for name, mod in pkgs.items():
    try:
        m = importlib.import_module(mod)
        v = getattr(m, '__version__', '?')
        print(f'  ✅ {name:20s} {v}')
    except ImportError:
        print(f'  ❌ {name:20s} NOT INSTALLED')
"
echo ''
echo "✅ All dependencies installed"

---
## ➄ Download Fish S2-Pro model (~9 GB)

Downloads the sharded S2-Pro checkpoint from HuggingFace to `/content/models/s2-pro`.

- **First run:** 5–15 minutes depending on Colab bandwidth
- **Subsequent runs:** instant (files are cached on disk)

In [ ]:
%%bash
set -e
source /content/venv/bin/activate

MODEL_DIR="/content/models/s2-pro"

# Skip if already fully downloaded (>5 files = config + weights + tokenizer)
if [ -d "$MODEL_DIR" ] && [ "$(ls -1 $MODEL_DIR 2>/dev/null | wc -l)" -gt 5 ]; then
    echo "✅ Model already present — skipping download."
    du -sh "$MODEL_DIR"
else
    echo '=== Downloading fishaudio/s2-pro ==='
    mkdir -p "$MODEL_DIR"
    huggingface-cli download fishaudio/s2-pro \
        --local-dir "$MODEL_DIR" \
        --local-dir-use-symlinks False
    echo ''
    echo '✅ Download complete.'
    du -sh "$MODEL_DIR"
fi
echo ''
ls -lh "$MODEL_DIR" | head -15

---
## ➅ Verify GPU + VRAM

Sanity check: CUDA is available and we have enough VRAM for S2-Pro (~10 GB BF16).

In [ ]:
%%bash
set -e
source /content/venv/bin/activate

python << 'PYEOF'
import torch

print(f'CUDA available : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    mem_total = props.total_memory / 1e9
    mem_free, _ = torch.cuda.mem_get_info()
    mem_free_gb = mem_free / 1e9

    print(f'GPU            : {props.name}')
    print(f'VRAM total     : {mem_total:.1f} GB')
    print(f'VRAM free      : {mem_free_gb:.1f} GB')

    if mem_free_gb < 10:
        print('\n⚠️  WARNING: Less than 10 GB free VRAM. S2-Pro may OOM.')
        print('   Try: Runtime → Disconnect and delete runtime → Reconnect with T4/A100')
    else:
        print('\n✅ Sufficient VRAM for S2-Pro')
else:
    print('\n❌ NO GPU DETECTED!')
    print('   Go to Runtime → Change runtime type → GPU (T4)')
    raise SystemExit(1)
PYEOF

---
## ➆ Write the FastAPI server module

This cell writes the complete `brainclip_colab_server.py` inline. It combines:
- Voice synthesis endpoints (`/voice/job`, `/voice/encode-ref`, `/voice/clone`)
- Video rendering endpoint (`/render`)
- Health check (`/health`)
- Whisper transcription

The file is written to `/content/notebooks/brainclip_colab_server.py` so uvicorn can import it.

In [ ]:
%%bash
set -e
mkdir -p /content/notebooks

# Check if the server module already exists from the cloned repo
if [ -f /content/notebooks/brainclip_colab_server.py ]; then
    echo '✅ Server module already exists from repo clone'
    wc -l /content/notebooks/brainclip_colab_server.py
else
    echo '❌ Server module not found — re-clone the repo or check the clone step'
    exit 1
fi

# Ensure the notebooks directory has an __init__.py for Python imports
touch /content/notebooks/__init__.py

echo ''
echo '=== Server endpoints ==='
grep -E '(@app\.(get|post|delete|put))' /content/notebooks/brainclip_colab_server.py | sed 's/^/  /'

---
## ➇ Start FastAPI server

Launches the uvicorn server in the background on port 8000 using the **venv Python directly**
(not `source activate`) to guarantee correct packages.

In [ ]:
%%bash
set -e
source /content/venv/bin/activate
export BRAINCLIP_APP_ROOT=/content
export PYTHONPATH=$PYTHONPATH:/content

# Kill any previous server on port 8000
pkill -f "uvicorn.*brainclip_colab_server" 2>/dev/null || true
sleep 2

# Quick import sanity check
echo '=== Import checks ==='
/content/venv/bin/python -c "
import vllm; print(f'  vLLM     : {vllm.__version__}')
import torch; print(f'  PyTorch  : {torch.__version__}')
import fastapi; print(f'  FastAPI  : {fastapi.__version__}')
" || { echo '❌ Import check failed! See install step.'; exit 1; }

echo ''
echo '=== Starting uvicorn server ==='
# Use the venv python directly for reliable imports
nohup /content/venv/bin/python -m uvicorn notebooks.brainclip_colab_server:app \
    --app-dir /content \
    --host 0.0.0.0 \
    --port 8000 \
    > /content/uvicorn.log 2>&1 &

# Wait for server to boot
echo 'Waiting for server to start...'
for i in $(seq 1 15); do
    if curl -s http://127.0.0.1:8000/health > /dev/null 2>&1; then
        echo ''
        echo '✅ Server is UP!'
        curl -s http://127.0.0.1:8000/health | python -m json.tool
        exit 0
    fi
    sleep 1
    printf '.'
done

echo ''
echo '❌ Server failed to start. Last 30 lines of log:'
tail -30 /content/uvicorn.log
exit 1

---
## ➈ Start Cloudflare Tunnel

Exposes the local FastAPI server to the internet via a free Cloudflare quick tunnel.

**→ Copy the printed URL into Brainclip → Settings → Colab Tunnel URL**

In [ ]:
%%bash
set -e

# Install cloudflared if not present
if ! command -v cloudflared &>/dev/null; then
    echo '=== Installing cloudflared ==='
    wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
        -O /usr/local/bin/cloudflared
    chmod +x /usr/local/bin/cloudflared
    echo "✅ cloudflared $(cloudflared --version 2>&1 | head -1)"
else
    echo "✅ cloudflared already installed"
fi

# Kill any existing tunnel
pkill -f "cloudflared tunnel" 2>/dev/null || true
sleep 2

# Start tunnel in background
echo '=== Starting Cloudflare Tunnel ==='
cloudflared tunnel --url http://127.0.0.1:8000 \
    --logfile /content/cloudflared.log \
    --metrics 0.0.0.0:9100 \
    > /content/cloudflared.out 2>&1 &

# Wait for tunnel URL to appear
echo 'Waiting for tunnel URL...'
for i in $(seq 1 20); do
    TUNNEL_URL=$(grep -o 'https://[-a-zA-Z0-9.]*trycloudflare.com' /content/cloudflared.out 2>/dev/null | head -n 1)
    if [ -n "$TUNNEL_URL" ]; then
        break
    fi
    sleep 1
    printf '.'
done

echo ''
echo ''

if [ -z "$TUNNEL_URL" ]; then
    echo '❌ Tunnel URL not found. Last 20 lines of log:'
    tail -20 /content/cloudflared.out
    exit 1
fi

echo '========================================================'
echo ''
echo "  🚀 TUNNEL URL: $TUNNEL_URL"
echo ''
echo '========================================================'
echo ''
echo 'Paste this URL into Brainclip → Settings → Colab Tunnel URL'
echo ''

# Verify tunnel is live
echo '=== Health check via tunnel ==='
curl -s "${TUNNEL_URL}/health" | python -m json.tool || echo '⚠️  Health check failed — wait 10s and try the URL manually'

---
## ➉ Reference audio cache

Check what reference audio files have been cached (for voice cloning).

In [ ]:
from pathlib import Path

REF_DIR = Path('/content/cache/refs')
REF_DIR.mkdir(parents=True, exist_ok=True)

files = list(REF_DIR.glob('*'))
print(f'Reference cache: {REF_DIR}')
print(f'Cached files   : {len(files)}')
for f in files:
    size_kb = f.stat().st_size / 1024
    print(f'  - {f.name}  ({size_kb:.1f} KB)')

if not files:
    print('  (empty — references will be cached after first voice job)')

---
## 🛠️ Utilities

### View server logs

In [ ]:
# Run this cell to see the last 50 lines of the server log
!tail -50 /content/uvicorn.log

### Restart server (without re-installing)

In [ ]:
%%bash
set -e
export BRAINCLIP_APP_ROOT=/content
export PYTHONPATH=$PYTHONPATH:/content

pkill -f "uvicorn.*brainclip_colab_server" 2>/dev/null || true
sleep 2

nohup /content/venv/bin/python -m uvicorn notebooks.brainclip_colab_server:app \
    --app-dir /content \
    --host 0.0.0.0 \
    --port 8000 \
    > /content/uvicorn.log 2>&1 &

sleep 5
curl -s http://127.0.0.1:8000/health | python -m json.tool && echo '✅ Server restarted' || echo '❌ Failed'

### Check GPU memory

In [ ]:
!nvidia-smi

### Test health endpoint locally

In [ ]:
import requests, json

try:
    r = requests.get('http://127.0.0.1:8000/health', timeout=5)
    data = r.json()
    print(json.dumps(data, indent=2))
    
    # Pretty status
    status = '✅' if data.get('status') == 'ok' else '⚠️'
    print(f'\n{status} Server status: {data.get("status")}')
    print(f'   Engine: {data.get("engine_type")}')
    print(f'   GPU VRAM free: {data.get("gpu_mem_free_gb")} GB')
except Exception as e:
    print(f'❌ Cannot reach server: {e}')

### Clear reference cache

In [ ]:
import requests
try:
    r = requests.delete('http://127.0.0.1:8000/voice/cache', timeout=5)
    print(r.json())
except Exception as e:
    print(f'Error: {e}')

---
## 📝 Operational Notes

### Architecture
```
Brainclip Web App  ───>  Cloudflare Tunnel  ───>  FastAPI (port 8000)
                                                    ├─ /health
                                                    ├─ /voice/job       → Fish S2-Pro TTS
                                                    ├─ /voice/encode-ref → Reference encoding
                                                    ├─ /voice/clone      → Voice cloning
                                                    └─ /render           → Remotion video render
```

### Common issues

| Issue | Solution |
|---|---|
| `model_oom` error | Restart runtime: Runtime → Disconnect and delete runtime → Run All |
| Tunnel URL stops working | Re-run cell ➇ (Start Cloudflare Tunnel) |
| Server not responding | Check logs: run the "View server logs" utility cell |
| `No GPU detected` | Runtime → Change runtime type → GPU (T4) |
| Packages broken after Colab update | Delete venv and re-run from cell ➂: `!rm -rf /content/venv` |

### File layout
```
/content/
├─ venv/                      ← Isolated Python environment
├─ models/s2-pro/             ← Fish S2-Pro model weights (~9 GB)
├─ cache/refs/                ← Cached speaker reference tokens
├─ notebooks/
│  └─ brainclip_colab_server.py ← FastAPI server module
├─ remotion/                  ← Remotion video templates
├─ uvicorn.log                ← Server log
└─ cloudflared.out            ← Tunnel log
```

### Keeping things running
- **Don’t close this tab** — the server and tunnel die when the Colab session ends
- Models stay loaded in GPU memory across jobs (no re-load per request)
- If GPU memory drops below 2 GB free, restart the runtime
- The tunnel URL changes each time you restart — update Brainclip settings